# Tiered Helmholtz Transfer Dataset (Rigorous, Real/Imag Diagnostics)

This notebook builds a scientifically structured dataset for three transfer tiers:

- Tier A: \(16 
ightarrow 32\)
- Tier B: \(32 
ightarrow 64\)
- Tier C: \(64 
ightarrow 128\)

Design rules:
- Fixed grid size: `n_tot = 500` for all tiers.
- Tier-specific PML from `omega_high` settings.
- Randomized point-source generation.
- Validation at each stage.
- Diagnostics use **real and imaginary components separately** (no complex-magnitude metrics).

In [ ]:
from __future__ import annotations

import csv
import time
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import scipy.sparse as sp
import scipy.sparse.linalg as spla
import matplotlib.pyplot as plt
from tqdm.auto import tqdm


## Configuration

This cell defines the PML table (from your existing setup), tier map, and run settings.

Source randomization:
- amount: 3 to 6
- location: inside non-PML interior with margin
- amplitude: 1.0 to 2.0

In [ ]:
PML_CONFIG = {
    16:  {"npml": 104, "eta": 70.0},
    32:  {"npml": 88,  "eta": 110.0},
    64:  {"npml": 72,  "eta": 190.0},
    128: {"npml": 60,  "eta": 320.0},
}

TIERS = [
    {"label": "Tier A", "omega_low": 16, "omega_high": 32, "purpose": "Baseline Calibration (Mastery Level)."},
    {"label": "Tier B", "omega_low": 32, "omega_high": 64, "purpose": "Complexity Test (Scaling properties)."},
    {"label": "Tier C", "omega_low": 64, "omega_high": 128, "purpose": "Stress Test (Numerical dispersion limit)."},
]

@dataclass
class RunCfg:
    seed: int = 42
    n_tot_target: int = 500
    n_train: int = 12
    n_val: int = 4
    n_test: int = 4
    n_src_min: int = 3
    n_src_max: int = 6
    source_margin: int = 16
    source_amp_min: float = 1.0
    source_amp_max: float = 2.0
    stencil_order: int = 2

run_cfg = RunCfg()
np.random.seed(run_cfg.seed)

print("run_cfg:", run_cfg)


## Geometry and PML Sanity Check

We verify that each tier has valid interior region on the fixed 500x500 grid and visualize PML damping profiles.

In [ ]:
def _pml_power(cfg: dict[str, float]) -> float:
    return float(cfg.get("pml_power", 2.0))


def sigma_profile_1d(n_tot: int, n_pml: int, eta: float, pml_power: float) -> np.ndarray:
    sig = np.zeros(n_tot, dtype=float)
    if n_pml <= 0:
        return sig

    for i in range(n_tot):
        if i < n_pml:
            dist = (n_pml - i) / n_pml
        elif i >= (n_tot - n_pml):
            dist = (i - (n_tot - n_pml) + 1) / n_pml
        else:
            dist = 0.0
        sig[i] = eta * (dist ** pml_power)
    return sig


def sigma_map_2d(n_tot: int, n_pml: int, eta: float, pml_power: float) -> np.ndarray:
    s = sigma_profile_1d(n_tot, n_pml, eta, pml_power)
    return s[:, None] + s[None, :]


def validate_tier_geometry(tiers, pml_config, n_tot):
    print(f"{'Tier':<8} {'ωlow':>6} {'ωhigh':>7} {'npml':>6} {'n_phys':>8}  Purpose")
    print("-" * 90)
    for tier in tiers:
        ol = int(tier["omega_low"])
        oh = int(tier["omega_high"])
        assert oh == 2 * ol, f"Expected octave jump, got {ol}->{oh}"
        npml = int(pml_config[oh]["npml"])
        n_phys = n_tot - 2 * npml
        assert n_phys > 2, f"Invalid interior size for {tier['label']}: {n_phys}"
        print(f"{tier['label']:<8} {ol:>6d} {oh:>7d} {npml:>6d} {n_phys:>8d}  {tier['purpose']}")


validate_tier_geometry(TIERS, PML_CONFIG, run_cfg.n_tot_target)

fig, axs = plt.subplots(2, 2, figsize=(12, 8), constrained_layout=True)
axs = axs.ravel()
for ax, om in zip(axs, [16, 32, 64, 128]):
    cfg = PML_CONFIG[om]
    npml = int(cfg["npml"])
    eta = float(cfg["eta"])
    p = _pml_power(cfg)
    n_tot = int(run_cfg.n_tot_target)
    sig = sigma_profile_1d(n_tot, npml, eta, p)

    ax.plot(sig, lw=2)
    ax.axvline(npml, color="k", ls="--", lw=1)
    ax.axvline(n_tot - npml - 1, color="k", ls="--", lw=1)
    ax.set_title(f"ω={om} | n_tot={n_tot}, npml={npml}, n_phys={n_tot-2*npml}, η={eta}")
    ax.set_xlabel("grid index")
    ax.set_ylabel("σ(i)")
    ax.grid(alpha=0.3)
plt.show()


## Helmholtz Assembly and Numerical Validation Helpers

This section defines the operator, source generator, and real/imag diagnostics used by the dataset pipeline.

In [ ]:
def get_helmholtz_matrix(
    *,
    omega: float,
    n_tot: int,
    n_pml: int,
    eta: float,
    pml_power: float = 2.0,
    stencil_order: int = 2,
):
    if stencil_order not in (2, 4):
        raise ValueError(f"stencil_order must be 2 or 4, got {stencil_order}")
    if n_tot < 5:
        raise ValueError(f"n_tot must be >= 5, got {n_tot}")
    if n_pml < 0:
        raise ValueError(f"n_pml must be >= 0, got {n_pml}")
    if 2 * n_pml >= n_tot - 2:
        raise ValueError(f"PML too thick: n_tot={n_tot}, n_pml={n_pml}")

    h = 1.0 / (n_tot - 1)
    sig = sigma_profile_1d(n_tot, n_pml, eta, pml_power)
    s = 1.0 / (1.0 + 1j * sig / (omega / (2.0 * np.pi)))

    rows, cols, vals = [], [], []

    def add(r, c, v):
        rows.append(r)
        cols.append(c)
        vals.append(v)

    def lin(i, j):
        return i * n_tot + j

    d2, o21 = -2.0, 1.0
    d4, o41, o42 = -2.5, 4.0 / 3.0, -1.0 / 12.0
    k2 = float(omega) * float(omega)

    for i in range(n_tot):
        sy2 = s[i] * s[i]
        for j in range(n_tot):
            r = lin(i, j)

            if i == 0 or j == 0 or i == n_tot - 1 or j == n_tot - 1:
                add(r, r, 1.0 + 0.0j)
                continue

            sx2 = s[j] * s[j]
            use_2nd = (
                stencil_order == 2
                or i < 2 or j < 2
                or i > n_tot - 3 or j > n_tot - 3
            )

            if use_2nd:
                add(r, r, ((sx2 * d2 + sy2 * d2) / (h * h)) + k2)
                add(r, lin(i, j - 1), sx2 * o21 / (h * h))
                add(r, lin(i, j + 1), sx2 * o21 / (h * h))
                add(r, lin(i - 1, j), sy2 * o21 / (h * h))
                add(r, lin(i + 1, j), sy2 * o21 / (h * h))
            else:
                add(r, r, ((sx2 * d4 + sy2 * d4) / (h * h)) + k2)
                add(r, lin(i, j - 1), sx2 * o41 / (h * h))
                add(r, lin(i, j + 1), sx2 * o41 / (h * h))
                add(r, lin(i, j - 2), sx2 * o42 / (h * h))
                add(r, lin(i, j + 2), sx2 * o42 / (h * h))
                add(r, lin(i - 1, j), sy2 * o41 / (h * h))
                add(r, lin(i + 1, j), sy2 * o41 / (h * h))
                add(r, lin(i - 2, j), sy2 * o42 / (h * h))
                add(r, lin(i + 2, j), sy2 * o42 / (h * h))

    return sp.coo_matrix((vals, (rows, cols)), shape=(n_tot * n_tot, n_tot * n_tot)).tocsr()


def make_rhs(n_tot: int, n_pml: int, rng: np.random.Generator, n_sources: int, margin: int, amp_min: float, amp_max: float):
    if n_sources < 1:
        raise ValueError("n_sources must be >= 1")

    f = np.zeros((n_tot, n_tot), dtype=np.complex128)
    lo = n_pml + margin
    hi = n_tot - n_pml - margin
    if hi <= lo:
        raise ValueError(f"source margin too large: n_tot={n_tot}, n_pml={n_pml}, margin={margin}")

    for _ in range(n_sources):
        y = int(rng.integers(lo, hi))
        x = int(rng.integers(lo, hi))
        amp = float(rng.uniform(amp_min, amp_max))
        phase = float(rng.uniform(0.0, 2 * np.pi))
        f[y, x] += amp * np.exp(1j * phase)
    return f


def to_2ch(u: np.ndarray) -> np.ndarray:
    return np.stack([u.real, u.imag], axis=0).astype(np.float32)


def channel_rms(z: np.ndarray) -> tuple[float, float]:
    return (
        float(np.sqrt(np.mean(np.real(z) ** 2))),
        float(np.sqrt(np.mean(np.imag(z) ** 2))),
    )


def split_rel_residual(A: sp.csr_matrix, u: np.ndarray, b: np.ndarray, eps: float = 1e-30):
    r = A @ u - b
    rr_re = np.linalg.norm(np.real(r)) / max(np.linalg.norm(np.real(b)), eps)
    rr_im = np.linalg.norm(np.imag(r)) / max(np.linalg.norm(np.imag(b)), eps)
    return float(rr_re), float(rr_im)


## Smoke Test (2 Sources per Tier)

This is a pre-generation gate:
- one sample per tier,
- exactly 2 sources,
- report residuals and RMS by channel,
- visualize real/imag parts for RHS and solutions.

In [ ]:
rng = np.random.default_rng(run_cfg.seed + 999)
n_tot = int(run_cfg.n_tot_target)

fig, axs = plt.subplots(len(TIERS), 6, figsize=(18, 4 * len(TIERS)), constrained_layout=True)
if len(TIERS) == 1:
    axs = np.array([axs])

for i, tier in enumerate(TIERS):
    ol = int(tier["omega_low"])
    oh = int(tier["omega_high"])
    cfg = PML_CONFIG[oh]
    n_pml = int(cfg["npml"])
    eta = float(cfg["eta"])
    pml_power = _pml_power(cfg)

    A_low = get_helmholtz_matrix(
        omega=float(ol),
        n_tot=n_tot,
        n_pml=n_pml,
        eta=eta,
        pml_power=pml_power,
        stencil_order=run_cfg.stencil_order,
    )
    A_high = get_helmholtz_matrix(
        omega=float(oh),
        n_tot=n_tot,
        n_pml=n_pml,
        eta=eta,
        pml_power=pml_power,
        stencil_order=run_cfg.stencil_order,
    )

    rhs = make_rhs(
        n_tot=n_tot,
        n_pml=n_pml,
        rng=rng,
        n_sources=2,
        margin=run_cfg.source_margin,
        amp_min=run_cfg.source_amp_min,
        amp_max=run_cfg.source_amp_max,
    )
    b = rhs.reshape(-1)

    solve_low = spla.factorized(A_low.tocsc())
    solve_high = spla.factorized(A_high.tocsc())

    u_low = solve_low(b).reshape(n_tot, n_tot)
    u_high = solve_high(b).reshape(n_tot, n_tot)

    rr_low_re, rr_low_im = split_rel_residual(A_low, u_low.reshape(-1), b)
    rr_high_re, rr_high_im = split_rel_residual(A_high, u_high.reshape(-1), b)

    ul_re, ul_im = channel_rms(u_low)
    uh_re, uh_im = channel_rms(u_high)

    print(
        f"{tier['label']} ({ol}->{oh}) | n_phys={n_tot - 2*n_pml} | "
        f"u_low_rms(re,im)=({ul_re:.3e},{ul_im:.3e}) | "
        f"u_high_rms(re,im)=({uh_re:.3e},{uh_im:.3e}) | "
        f"relres_low(re,im)=({rr_low_re:.2e},{rr_low_im:.2e}) | "
        f"relres_high(re,im)=({rr_high_re:.2e},{rr_high_im:.2e})"
    )

    panels = [
        (np.real(rhs), "RHS Re"),
        (np.imag(rhs), "RHS Im"),
        (np.real(u_low), f"u_low Re (ω={ol})"),
        (np.imag(u_low), f"u_low Im (ω={ol})"),
        (np.real(u_high), f"u_high Re (ω={oh})"),
        (np.imag(u_high), f"u_high Im (ω={oh})"),
    ]

    for j, (arr, title) in enumerate(panels):
        im = axs[i, j].imshow(arr, origin="lower", cmap="coolwarm")
        axs[i, j].set_title(f"{tier['label']} | {title}")
        axs[i, j].set_xticks([])
        axs[i, j].set_yticks([])
        plt.colorbar(im, ax=axs[i, j], fraction=0.046)

plt.show()


## Tiered Dataset Builder

Each sample is generated with shared RHS and solved at low/high frequencies.
Saved tensors:
- `X_up = u_low`, `Y_up = u_high`
- `X_down = u_high`, `Y_down = u_low`

Diagnostics are all real/imag separated.

In [ ]:
def build_tier_split_dataset(tier, run_cfg: RunCfg, n_samples: int, seed_offset: int = 0):
    rng = np.random.default_rng(run_cfg.seed + seed_offset)

    ol = int(tier["omega_low"])
    oh = int(tier["omega_high"])
    pml_cfg = PML_CONFIG[oh]

    n_tot = int(run_cfg.n_tot_target)
    n_pml = int(pml_cfg["npml"])
    eta = float(pml_cfg["eta"])
    pml_power = _pml_power(pml_cfg)

    A_low = get_helmholtz_matrix(
        omega=float(ol),
        n_tot=n_tot,
        n_pml=n_pml,
        eta=eta,
        pml_power=pml_power,
        stencil_order=run_cfg.stencil_order,
    )
    A_high = get_helmholtz_matrix(
        omega=float(oh),
        n_tot=n_tot,
        n_pml=n_pml,
        eta=eta,
        pml_power=pml_power,
        stencil_order=run_cfg.stencil_order,
    )

    solve_low = spla.factorized(A_low.tocsc())
    solve_high = spla.factorized(A_high.tocsc())

    X_up, Y_up, X_down, Y_down = [], [], [], []
    diagnostics = []

    t0 = time.perf_counter()
    for k in tqdm(range(n_samples), desc=f"{tier['label']} build", leave=False):
        nsrc = int(rng.integers(run_cfg.n_src_min, run_cfg.n_src_max + 1))
        rhs = make_rhs(
            n_tot=n_tot,
            n_pml=n_pml,
            rng=rng,
            n_sources=nsrc,
            margin=run_cfg.source_margin,
            amp_min=run_cfg.source_amp_min,
            amp_max=run_cfg.source_amp_max,
        )
        b = rhs.reshape(-1)

        u_low = solve_low(b).reshape(n_tot, n_tot)
        u_high = solve_high(b).reshape(n_tot, n_tot)

        X_up.append(to_2ch(u_low))
        Y_up.append(to_2ch(u_high))
        X_down.append(to_2ch(u_high))
        Y_down.append(to_2ch(u_low))

        rr_low_re, rr_low_im = split_rel_residual(A_low, u_low.reshape(-1), b)
        rr_high_re, rr_high_im = split_rel_residual(A_high, u_high.reshape(-1), b)

        rhs_rms_re, rhs_rms_im = channel_rms(rhs)
        ul_rms_re, ul_rms_im = channel_rms(u_low)
        uh_rms_re, uh_rms_im = channel_rms(u_high)

        diagnostics.append({
            "sample": int(k),
            "n_sources": int(nsrc),
            "rhs_rms_re": rhs_rms_re,
            "rhs_rms_im": rhs_rms_im,
            "u_low_rms_re": ul_rms_re,
            "u_low_rms_im": ul_rms_im,
            "u_high_rms_re": uh_rms_re,
            "u_high_rms_im": uh_rms_im,
            "relres_low_re": rr_low_re,
            "relres_low_im": rr_low_im,
            "relres_high_re": rr_high_re,
            "relres_high_im": rr_high_im,
        })

    elapsed = time.perf_counter() - t0

    return {
        "tier": tier,
        "n_tot": n_tot,
        "n_pml": n_pml,
        "eta": eta,
        "X_up": np.stack(X_up, axis=0),
        "Y_up": np.stack(Y_up, axis=0),
        "X_down": np.stack(X_down, axis=0),
        "Y_down": np.stack(Y_down, axis=0),
        "diagnostics": diagnostics,
        "build_seconds": float(elapsed),
        "seconds_per_sample": float(elapsed / max(n_samples, 1)),
    }


## Full Build: Train/Val/Test Across Tiers

Split seeds are separated to prevent overlap across train/val/test and tiers.

In [ ]:
split_plan = [
    ("train", run_cfg.n_train, 0),
    ("val", run_cfg.n_val, 10_000),
    ("test", run_cfg.n_test, 20_000),
]

dataset = {"config": run_cfg, "tiers": {}}

t_all = time.perf_counter()
for tier in TIERS:
    label = tier["label"]
    dataset["tiers"][label] = {}

    for split_name, n_samples, split_seed_offset in split_plan:
        t0 = time.perf_counter()
        block = build_tier_split_dataset(
            tier,
            run_cfg,
            n_samples=n_samples,
            seed_offset=split_seed_offset + 1000 * (TIERS.index(tier) + 1),
        )
        dataset["tiers"][label][split_name] = block

        dt = time.perf_counter() - t0
        print(
            f"{label} | {split_name}: n={n_samples} | elapsed={dt:.2f}s | "
            f"per_sample={block['seconds_per_sample']:.3f}s"
        )

print(f"Total tiered dataset build time: {time.perf_counter() - t_all:.2f}s")


## Validation Summary (Real/Imag Only)

Reports per tier and split:
- residual statistics by channel,
- field RMS statistics by channel,
- identity baseline MSE per channel (`u_high` vs `u_low`).

In [ ]:
def summarize_block(block):
    d = block["diagnostics"]

    rlr = np.array([x["relres_low_re"] for x in d], dtype=float)
    rli = np.array([x["relres_low_im"] for x in d], dtype=float)
    rhr = np.array([x["relres_high_re"] for x in d], dtype=float)
    rhi = np.array([x["relres_high_im"] for x in d], dtype=float)

    ulr = np.array([x["u_low_rms_re"] for x in d], dtype=float)
    uli = np.array([x["u_low_rms_im"] for x in d], dtype=float)
    uhr = np.array([x["u_high_rms_re"] for x in d], dtype=float)
    uhi = np.array([x["u_high_rms_im"] for x in d], dtype=float)

    Xup = block["X_up"]
    Yup = block["Y_up"]
    mse_re = float(np.mean((Xup[:, 0] - Yup[:, 0]) ** 2))
    mse_im = float(np.mean((Xup[:, 1] - Yup[:, 1]) ** 2))

    return {
        "n": len(d),
        "rlr_med": float(np.median(rlr)),
        "rlr_max": float(np.max(rlr)),
        "rli_med": float(np.median(rli)),
        "rli_max": float(np.max(rli)),
        "rhr_med": float(np.median(rhr)),
        "rhr_max": float(np.max(rhr)),
        "rhi_med": float(np.median(rhi)),
        "rhi_max": float(np.max(rhi)),
        "ulr_med": float(np.median(ulr)),
        "uli_med": float(np.median(uli)),
        "uhr_med": float(np.median(uhr)),
        "uhi_med": float(np.median(uhi)),
        "sec_per_sample": float(block["seconds_per_sample"]),
        "id_mse_re": mse_re,
        "id_mse_im": mse_im,
    }

print("Validation summary")
print("-" * 190)
print(
    f"{'Tier':<8} {'Split':<7} {'n':>3} "
    f"{'rrL_re_med':>11} {'rrL_re_max':>11} {'rrL_im_med':>11} {'rrL_im_max':>11} "
    f"{'rrH_re_med':>11} {'rrH_re_max':>11} {'rrH_im_med':>11} {'rrH_im_max':>11} "
    f"{'uL_re_med':>10} {'uL_im_med':>10} {'uH_re_med':>10} {'uH_im_med':>10} "
    f"{'sec/s':>8} {'id_mse_re':>11} {'id_mse_im':>11}"
)
print("-" * 190)

for tier in TIERS:
    label = tier["label"]
    for split_name, _, _ in split_plan:
        s = summarize_block(dataset["tiers"][label][split_name])
        print(
            f"{label:<8} {split_name:<7} {s['n']:>3d} "
            f"{s['rlr_med']:>11.2e} {s['rlr_max']:>11.2e} {s['rli_med']:>11.2e} {s['rli_max']:>11.2e} "
            f"{s['rhr_med']:>11.2e} {s['rhr_max']:>11.2e} {s['rhi_med']:>11.2e} {s['rhi_max']:>11.2e} "
            f"{s['ulr_med']:>10.2e} {s['uli_med']:>10.2e} {s['uhr_med']:>10.2e} {s['uhi_med']:>10.2e} "
            f"{s['sec_per_sample']:>8.3f} {s['id_mse_re']:>11.2e} {s['id_mse_im']:>11.2e}"
        )


## Persist Data + CSV Manifests

Storage layout:
- `experiments/data/tiered_omega_transfer_rigorous_500/`
- per tier folder: `TierA_16_32`, `TierB_32_64`, `TierC_64_128`
- each tier folder: `train.npz`, `val.npz`, `test.npz`, `manifest.csv`
- global file: `index.csv`

In [ ]:
root = Path("experiments/data/tiered_omega_transfer_rigorous_500")
root.mkdir(parents=True, exist_ok=True)

global_rows = []

for tier in TIERS:
    label = tier["label"]
    ol = int(tier["omega_low"])
    oh = int(tier["omega_high"])
    tier_slug = f"{label.replace(' ', '')}_{ol}_{oh}"
    tier_dir = root / tier_slug
    tier_dir.mkdir(parents=True, exist_ok=True)

    tier_rows = []

    for split_name, _, _ in split_plan:
        block = dataset["tiers"][label][split_name]
        npz_path = tier_dir / f"{split_name}.npz"

        np.savez_compressed(
            npz_path,
            X_up=block["X_up"],
            Y_up=block["Y_up"],
            X_down=block["X_down"],
            Y_down=block["Y_down"],
            n_tot=np.array([block["n_tot"]], dtype=np.int32),
            n_pml=np.array([block["n_pml"]], dtype=np.int32),
            eta=np.array([block["eta"]], dtype=np.float64),
        )

        diag_csv = tier_dir / f"diagnostics_{split_name}.csv"
        with open(diag_csv, "w", newline="", encoding="utf-8") as f:
            w = csv.DictWriter(f, fieldnames=list(block["diagnostics"][0].keys()))
            w.writeheader()
            w.writerows(block["diagnostics"])

        row = {
            "tier": label,
            "omega_low": ol,
            "omega_high": oh,
            "split": split_name,
            "n_samples": int(block["X_up"].shape[0]),
            "n_tot": int(block["n_tot"]),
            "n_pml": int(block["n_pml"]),
            "n_phys": int(block["n_tot"] - 2 * block["n_pml"]),
            "eta": float(block["eta"]),
            "seconds_per_sample": float(block["seconds_per_sample"]),
            "npz_path": str(npz_path),
            "diagnostics_csv": str(diag_csv),
        }
        tier_rows.append(row)
        global_rows.append(row)

    tier_manifest = tier_dir / "manifest.csv"
    with open(tier_manifest, "w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=list(tier_rows[0].keys()))
        w.writeheader()
        w.writerows(tier_rows)

index_path = root / "index.csv"
with open(index_path, "w", newline="", encoding="utf-8") as f:
    w = csv.DictWriter(f, fieldnames=list(global_rows[0].keys()))
    w.writeheader()
    w.writerows(global_rows)

print("Saved dataset tree at:", root)
print("Global index:", index_path)


In [4]:
!find . -type f \( -name '*.pth' -o -name '*.pt' \) | sort


./checkpoints/cnn_16_32.pth
./checkpoints/cnn_32_64.pth
./checkpoints/cnn_64_128.pth
./checkpoints/unet_16_32.pth
./checkpoints/unet_32_64.pth
./checkpoints/unet_64_128.pth
./experiments/checkpoints/T_down_gemidrie.pth
./experiments/checkpoints/T_down_overnight_best.pth
./experiments/checkpoints/T_up_gemidrie.pth
./experiments/checkpoints/T_up_overnight_best.pth
./experiments/models/transfer_operator_generations/TierA_down_Gen2_SimpleCNN.pth
./experiments/models/transfer_operator_generations/TierA_up_Gen2_SimpleCNN.pth
./experiments/models/transfer_operator_generations/TierB_down_Gen2_SimpleCNN.pth
./experiments/models/transfer_operator_generations/TierB_up_Gen2_SimpleCNN.pth
./experiments/models/transfer_operator_generations/TierC_down_Gen2_SimpleCNN.pth
./experiments/models/transfer_operator_generations/TierC_up_Gen2_SimpleCNN.pth
./FreqTransfer_Ladder_0211_1859/model_16_32.pth
./FreqTransfer_Ladder_0211_1859/model_32_64.pth
./FreqTransfer_Ladder_0211_1859/model_64_128.pth
./helmholt

In [5]:
from pathlib import Path
import torch, os

print("cwd:", os.getcwd())

cands = sorted(Path(".").rglob("TierA_up_Gen2_SimpleCNN.pth"))
print("matches:", len(cands))
for p in cands:
    print(p)

if cands:
    p = cands[0]
    state = torch.load(p, map_location="cpu")
    print("loaded:", p)
    print("type:", type(state), "keys:", len(state))


cwd: /math/home/fkiewiet/Freq2Transfer/experiments
matches: 1
experiments/models/transfer_operator_generations/TierA_up_Gen2_SimpleCNN.pth
loaded: experiments/models/transfer_operator_generations/TierA_up_Gen2_SimpleCNN.pth
type: <class 'collections.OrderedDict'> keys: 6


In [6]:
p = "experiments/models/transfer_operator_generations/TierA_up_Gen2_SimpleCNN.pth"


## Notes

- This version removes duplicate/overlapping pipelines and keeps one coherent path.
- All diagnostics are real/imag split.
- The smoke-test cell should be run before the full build cell.

## 1) Validation Scope and Criteria

We validate the trained Gen2 weights from multiple perspectives:

1. **Integrity**: files exist, load correctly, parameter sanity.
2. **Predictive accuracy**: test-set metrics (Re/Im), compared to identity baseline.
3. **Reliability**: per-sample error distributions and worst-case behavior.
4. **Physical plausibility proxy**: performance versus source complexity (`n_sources`).
5. **Robustness**: sensitivity to small input perturbations.
6. **Reporting**: save all tables for traceability.
